In [1]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pandas as pd
import json

import sys
from pathlib import Path

# Ajouter la racine du projet au PYTHONPATH
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data.openagenda_import import (
    get_toulouse_agendas,
    get_events_from_agenda,
    get_toulouse_events
)

In [2]:
events = get_toulouse_events()

100 agendas récupérés sur cette page - total actuel : 100 - status=200 
100 agendas récupérés sur cette page - total actuel : 200 - status=200 
100 agendas récupérés sur cette page - total actuel : 300 - status=200 
100 agendas récupérés sur cette page - total actuel : 400 - status=200 
100 agendas récupérés sur cette page - total actuel : 500 - status=200 
100 agendas récupérés sur cette page - total actuel : 600 - status=200 
100 agendas récupérés sur cette page - total actuel : 700 - status=200 
100 agendas récupérés sur cette page - total actuel : 800 - status=200 
33 agendas récupérés sur cette page - total actuel : 833 - status=200 
0 agendas récupérés sur cette page - total actuel : 833 - status=200 

Total : 833 agendas liés à Toulouse

[1/833] Agenda : Indépendance Day 3
0 événement(s) récupéré(s)- total actuel : 0
Agenda 8972891 : 0 événement(s) au total

→ 0 événement(s) récupéré(s) dans cet agenda
→ Total cumulé : 0 événement(s)

[2/833] Agenda : OpenAgenda et les événement

In [3]:
df = pd.DataFrame(events)

print(df.shape)
df.head()

(21916, 190)


,longDescription,country,featured,private,keywords,accessibility,dateRange,timezone,imageCredits,originAgenda,...,musique-du-monde,genre,category-group,titre-image,mail,type-structure,payant,lien-site,creditsimage,tag-group-5
0,☀️ Les Quartiers d’Été 2026 — Toulouse les Org...,France (Métropole),False,0,"[orgue, festival, musique, patrimoine, concert...","{'ii': True, 'hi': True, 'vi': True, 'pi': Tru...",15 juillet - 19 septembre,Europe/Paris,NaN,"{'uid': 35944609, 'image': 'agenda35944609.jpg...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,**INSCRIPTIONS CLOSES**\n\nProgramme complet e...,France (Métropole),False,0,"[structuration, développement, musiques actuel...","{'ii': False, 'hi': False, 'vi': False, 'pi': ...",5 octobre 2026 - 13 janvier 2027,Europe/Paris,NaN,"{'uid': 94600341, 'image': 'f2a400882b314f22b8...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,De nouvelles vibrations musicales arrivent à T...,France (Métropole),False,0,"[orgues, festival, musique]","{'ii': True, 'hi': True, 'vi': True, 'pi': Tru...",7 - 18 octobre,Europe/Paris,NaN,"{'uid': 35944609, 'image': 'agenda35944609.jpg...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,🔵 Festival international Toulouse les Orgues #...,France (Métropole),False,0,"[Musique classique, Concert orgue et chœur]","{'ii': False, 'hi': False, 'vi': False, 'pi': ...","Jeudi 8 octobre, 20h30",Europe/Paris,NaN,"{'uid': 35944609, 'image': '0355f66326084b2aac...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,🔵 Festival international Toulouse les Orgues #...,France (Métropole),False,0,"[Musiques ambiantes, , Nuits du Gesù, Orgue ex...","{'ii': False, 'hi': False, 'vi': False, 'pi': ...","Jeudi 8 octobre, 22h30",Europe/Paris,NaN,"{'uid': 35944609, 'image': '0355f66326084b2aac...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Supprimer les doublons 

In [4]:

df_doublons = df[df.duplicated(subset="uid", keep=False)].copy()

print("Nombre total de lignes :", len(df))
print("Nombre de UID uniques :", df["uid"].nunique())
print("Nombre de lignes avec UID dupliqué :", len(df_doublons))
print("Nombre de UID concernés :", df_doublons["uid"].nunique())

Nombre total de lignes : 21916
Nombre de UID uniques : 7203
Nombre de lignes avec UID dupliqué : 20607
Nombre de UID concernés : 5894


In [5]:
resultats_conflits = []

for colonne in df.columns:
    if colonne == "uid":
        continue

    nb_uid_en_conflit = 0

    for uid, groupe in df_doublons.groupby("uid"):
        valeurs = set()

        for valeur in groupe[colonne]:
            # Valeur manquante
            if valeur is None:
                continue

            if not isinstance(valeur, (list, dict)):
                try:
                    if pd.isna(valeur):
                        continue
                except (TypeError, ValueError):
                    pass

            # Liste ou dictionnaire rendu comparable
            if isinstance(valeur, (list, dict)):
                valeur = json.dumps(
                    valeur,
                    sort_keys=True,
                    ensure_ascii=False,
                    default=str
                )
            else:
                valeur = str(valeur).strip()

            if valeur != "":
                valeurs.add(valeur)

        if len(valeurs) > 1:
            nb_uid_en_conflit += 1

    resultats_conflits.append({
        "colonne": colonne,
        "uid_avec_valeurs_differentes": nb_uid_en_conflit,
        "pct_uid_en_conflit": round(
            nb_uid_en_conflit
            / df_doublons["uid"].nunique()
            * 100,
            2
        )
    })

conflits = (
    pd.DataFrame(resultats_conflits)
    .sort_values(
        "uid_avec_valeurs_differentes",
        ascending=False
    )
    .reset_index(drop=True)
)

display(conflits.head(40))

,colonne,uid_avec_valeurs_differentes,pct_uid_en_conflit
0,agenda_uid,5894,100.00
1,agenda_title,5894,100.00
2,addMethod,5883,99.81
3,sourceAgendas,5799,98.39
4,originAgenda,5127,86.99
5,updatedAt,4503,76.40
6,image,3677,62.39
7,location,626,10.62
8,valid,167,2.83
9,featured,30,0.51


In [ ]:
# Supression des doublons en gardant les infos les plus recentes 

df["updatedAt"] = pd.to_datetime(
    df["updatedAt"],
    errors="coerce",
    utc=True
)

lignes_fusionnees = []

for uid, groupe in df.groupby("uid", sort=False):

    # La version la plus récente sert de base
    groupe = groupe.sort_values("updatedAt", ascending=False)
    ligne = groupe.iloc[0].copy()

    for colonne in df.columns:
        if colonne == "uid":
            continue

        valeurs = groupe[colonne].tolist()

        # Valeur actuelle de la ligne principale
        valeur_base = ligne[colonne]

        # LISTES : réunir toutes les valeurs uniques
        if any(isinstance(v, list) for v in valeurs):
            elements_uniques = {}

            for valeur in valeurs:
                if not isinstance(valeur, list):
                    continue

                for element in valeur:
                    cle = json.dumps(
                        element,
                        sort_keys=True,
                        ensure_ascii=False,
                        default=str
                    )
                    elements_uniques[cle] = element

            ligne[colonne] = list(elements_uniques.values())

        # DICTIONNAIRES : compléter les clés manquantes
        elif any(isinstance(v, dict) for v in valeurs):
            dictionnaire_fusionne = {}

            # Parcours inversé pour que la version récente
            # remplace les anciennes valeurs en cas de conflit
            for valeur in reversed(valeurs):
                if isinstance(valeur, dict):
                    dictionnaire_fusionne.update({
                        cle: contenu
                        for cle, contenu in valeur.items()
                        if contenu not in [None, "", [], {}]
                    })

            ligne[colonne] = dictionnaire_fusionne or None

        # VALEURS SIMPLES : remplir seulement si la base est vide
        else:
            base_vide = (
                valeur_base is None
                or (
                    isinstance(valeur_base, str)
                    and valeur_base.strip() == ""
                )
                or (
                    not isinstance(valeur_base, (list, dict))
                    and pd.isna(valeur_base)
                )
            )

            if base_vide:
                for valeur in valeurs:
                    if valeur is None:
                        continue

                    if isinstance(valeur, str) and valeur.strip() == "":
                        continue

                    try:
                        if pd.isna(valeur):
                            continue
                    except (TypeError, ValueError):
                        pass

                    ligne[colonne] = valeur
                    break

    # Conserver explicitement tous les agendas
    ligne["agenda_uids"] = list(
        dict.fromkeys(
            groupe["agenda_uid"].dropna().tolist()
        )
    )

    ligne["agenda_titles"] = list(
        dict.fromkeys(
            titre
            for titre in groupe["agenda_title"].dropna().tolist()
            if str(titre).strip()
        )
    )

    lignes_fusionnees.append(ligne)

df_clean = pd.DataFrame(lignes_fusionnees).reset_index(drop=True)

# Suppresion des colonnes vides

In [7]:
# Suppression des liste vides 
for col in df_clean.columns:
    df_clean[col] = df_clean[col].apply(
        lambda x: None if isinstance(x, list) and len(x) == 0 else x
    )

In [8]:
# suppression des dictionaires vides
for col in df_clean.columns:
    df_clean[col] = df_clean[col].apply(
        lambda x: None if isinstance(x, dict) and len(x) == 0 else x
    )

In [9]:
taux_manquant = (
    df_clean.isna()
    .mean()
    .sort_values(ascending=False)
)

colonnes_95 = taux_manquant[taux_manquant >= 0.95]

display(colonnes_95)

type-de-pratique-veuillez-preciser       1.000000
emaillll                                 1.000000
motive                                   1.000000
type_devenements                         1.000000
mail                                     1.000000
                                           ...   
conditions-de-participation              0.964876
categories-wwwmuseumchamp-obligatoire    0.962099
type-de-public                           0.955713
thematiques-conservatoire                0.955713
public-concerne                          0.955435
Length: 144, dtype: float64

In [ ]:
# Colonnes à supprimer
colonnes_a_supprimer = colonnes_95.index.tolist()

# Suppression des colones a plus de 95% manquant
df_clean = df_clean.drop(columns=colonnes_a_supprimer)

In [11]:
for col in df_clean.columns:
    print(f"\n===== {col} =====")
    print(df_clean[col].value_counts(dropna=False))


===== longDescription =====
longDescription
NaN                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          564
Oubliez les supports d’écriture modernes, cette animation vous fera découvrir tous les outils pour écrire dans l’Antiquité grâce à des reconstitutions d’objets que vous pourrez manipuler.  \n.....  \nPhoto : Lydia Mouysset/Musée Saint-Raymond - Licence ouverte                                                                                                                                                                                      

In [12]:
colonnes_a_supprimer = [
    # Métadonnées OpenAgenda
    "state",
    "draft",
    "private",
    "valid",
    "featured",
    "addMethod",

    # Identifiants techniques
    "creatorUid",
    "ownerUid",
    "slug",

    # Informations sur les agendas
    "agenda_uid",
    "agenda_title",
    "originAgenda",
    "sourceAgendas",

    # Informations constantes
    "country",
    "timezone",

    # Codes non interprétables (pas de table de correspondance)
    "participation",
    "organisateur",
    "public",
    "types-devenements",
    "thematiques-metropolitaines",
    "evenement-ponctuel",
    "categories-wwwhttpswwwbibliothequetoulousefr-champ-obligatoire",
    "categories-wwwmsr",
    "categories-wwwmuseumchamp-obligatoire",
    "type-de-public",
    "public-concerne",
    "thematiques-conservatoire",
    "mediations-animations",
    "programme",
    "expositions",
    "patrimoine-tourisme",
    "spectacles",
    "conditions-dacces-publics",
    "marqueurs",

]

df_clean.drop(columns=colonnes_a_supprimer, inplace=True, errors="ignore")

In [13]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 7203 entries, 0 to 7202
Data columns (total 25 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   longDescription  6639 non-null   str                
 1   keywords         3856 non-null   object             
 2   accessibility    7203 non-null   object             
 3   dateRange        7203 non-null   str                
 4   imageCredits     2846 non-null   str                
 5   description      7185 non-null   str                
 6   title            7185 non-null   str                
 7   createdAt        7203 non-null   str                
 8   uid              7203 non-null   int64              
 9   timings          7203 non-null   object             
 10  firstTiming      7203 non-null   object             
 11  links            1649 non-null   object             
 12  updatedAt        7203 non-null   datetime64[us, UTC]
 13  image            7058 non-nul

# identification des informations

## handicaps

In [17]:
# handicaps 
mapping = {
    "hi": "handicap auditif",
    "vi": "handicap visuel",
    "pi": "handicap psychique",
    "mi": "handicap moteur",
    "ii": "handicap intellectuel"
}

df_clean["accessibility_text"] = df_clean["accessibility"].apply(
    lambda d: ", ".join(
        mapping[code]
        for code, present in d.items()
        if present
    ) if any(d.values()) else "Aucune information d'accessibilité"
)

In [18]:
df_clean.drop(columns="accessibility", inplace=True)

## modes de participations

In [ ]:
mapping = {
    1: "Présentiel",
    2: "En ligne",
    3: "Hybride"
}

df_clean["attendanceMode"] = df_clean["attendanceMode"].map(mapping)


In [20]:
df_clean.rename(columns={"attendanceMode": "mode_participation"}, inplace=True)

## statut

In [21]:
mapping = {
    1: "Programmé",
    2: "Reprogrammé",
    3: "Déplacé en ligne",
    4: "Reporté",
    5: "Complet",
    6: "Annulé"
}

df_clean["status"] = df_clean["status"].map(mapping)

In [22]:
df_clean["status"].value_counts()

status
Programmé      7067
Complet          67
Annulé           50
Reprogrammé      16
Reporté           3
Name: count, dtype: int64

# Nettoyage Formats 

In [ ]:
# nettoyage format longdescription
df_clean["longDescription"] = (
    df_clean["longDescription"]
    .str.replace(r"<[^>]+>", " ", regex=True)   # supprime les balises HTML
    .str.replace(r"[-_]{3,}", " ", regex=True)  # supprime les -------
    .str.replace(r"\s+", " ", regex=True)       # supprime les retours à la lignes 
    .str.strip()
)

In [28]:
# nettoyage liste 
df_clean["keywords"] = df_clean["keywords"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else x
)

df_clean["agenda_titles"] = df_clean["agenda_titles"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else x
)

### timings

In [31]:
df_clean["timings"] = df_clean["timings"].apply(
    lambda liste: " | ".join(
        f"Du {pd.to_datetime(creneau['begin']).strftime('%d/%m/%Y à %Hh%M')} "
        f"au {pd.to_datetime(creneau['end']).strftime('%d/%m/%Y à %Hh%M')}"
        for creneau in liste
    )
    if isinstance(liste, list)
    else None
)

In [30]:
for colonne in ["firstTiming", "lastTiming", "nextTiming"]:

    df_clean[colonne] = df_clean[colonne].apply(
        lambda x: (
            f"Du {pd.to_datetime(x['begin']).strftime('%d/%m/%Y à %Hh%M')} "
            f"au {pd.to_datetime(x['end']).strftime('%d/%m/%Y à %Hh%M')}"
        )
        if isinstance(x, dict)
        else None
    )

In [32]:
# Image

df_clean["image"] = df_clean["image"].apply(
    lambda x: (
        x["base"] + next(
            v["filename"] for v in x["variants"] if v["type"] == "full"
        )
        if isinstance(x, dict)
        else None
    )
)

In [33]:
# information de contact
df_clean["registration"] = df_clean["registration"].apply(
    lambda x: " | ".join(
        f"{item['type'].capitalize()} : {item['value']}"
        for item in x
    )
    if isinstance(x, list)
    else None
)

In [34]:
# lieux
df_clean["location_name"] = df_clean["location"].apply(
    lambda x: x.get("name") if isinstance(x, dict) else None
)

df_clean["location_address"] = df_clean["location"].apply(
    lambda x: x.get("address") if isinstance(x, dict) else None
)

df_clean["location_city"] = df_clean["location"].apply(
    lambda x: x.get("city") if isinstance(x, dict) else None
)

df_clean["location_postal_code"] = df_clean["location"].apply(
    lambda x: x.get("postalCode") if isinstance(x, dict) else None
)

df_clean["location_department"] = df_clean["location"].apply(
    lambda x: x.get("department") if isinstance(x, dict) else None
)

df_clean["location_region"] = df_clean["location"].apply(
    lambda x: x.get("region") if isinstance(x, dict) else None
)

df_clean["location_latitude"] = df_clean["location"].apply(
    lambda x: x.get("latitude") if isinstance(x, dict) else None
)

df_clean["location_longitude"] = df_clean["location"].apply(
    lambda x: x.get("longitude") if isinstance(x, dict) else None
)

In [ ]:
# homogénésition
df_clean["location_region"] = (
    df_clean["location_region"]
    .str.strip()
    .str.lower()
    .replace({
        "occitanie": "Occitanie",
        "occitania": "Occitanie"
    })
)

df_clean["location_department"] = (
    df_clean["location_department"]
    .str.strip()
    .str.lower()
    .replace({
        "haute-garonne": "Haute-Garonne",
        "alto garona": "Haute-Garonne"
    })
)

In [35]:
# description lieu
df_clean["location_text"] = df_clean["location"].apply(
    lambda x: " | ".join(
        partie
        for partie in [
            f"Lieu : {x.get('name')}" if x.get("name") else None,
            f"Adresse : {x.get('address')}" if x.get("address") else None,
            (
                f"Description du lieu : "
                f"{x.get('description', {}).get('fr')}"
                if isinstance(x.get("description"), dict)
                and x.get("description", {}).get("fr")
                else (
                    f"Description du lieu : {x.get('description')}"
                    if isinstance(x.get("description"), str)
                    else None
                )
            ),
            (
                f"Accès : {x.get('access', {}).get('fr')}"
                if isinstance(x.get("access"), dict)
                and x.get("access", {}).get("fr")
                else (
                    f"Accès : {x.get('access')}"
                    if isinstance(x.get("access"), str)
                    else None
                )
            ),
            f"Téléphone : {x.get('phone')}" if x.get("phone") else None,
            f"Email : {x.get('email')}" if x.get("email") else None,
            f"Site web : {x.get('website')}" if x.get("website") else None,
            f"Image du lieu : {x.get('image')}" if x.get("image") else None,
        ]
        if partie is not None
    )
    if isinstance(x, dict)
    else None
)

In [37]:
df_clean = df_clean.drop(columns=["location"])

In [38]:
# age
df_clean["age"] = df_clean["age"].apply(
    lambda x: (
        f"De {x['min']} à {x['max']} ans"
        if isinstance(x, dict)
        else None
    )
)

In [39]:
# lien
df_clean["links"] = df_clean["links"].apply(
    lambda liens: " | ".join(
        texte
        for item in liens
        if isinstance(item, dict)
        for texte in [
            f"Lien : {item.get('link')}"
            if item.get("link")
            else None,

            f"Titre : {item.get('data', {}).get('title')}"
            if isinstance(item.get("data"), dict)
            and item["data"].get("title")
            else None,

            f"Auteur : {item.get('data', {}).get('author')}"
            if isinstance(item.get("data"), dict)
            and item["data"].get("author")
            else None,

            f"Source : {item.get('data', {}).get('provider_name')}"
            if isinstance(item.get("data"), dict)
            and item["data"].get("provider_name")
            else None,

            f"Description : {item.get('data', {}).get('description')}"
            if isinstance(item.get("data"), dict)
            and item["data"].get("description")
            else None,

            f"Miniature : {item.get('data', {}).get('thumbnail_url')}"
            if isinstance(item.get("data"), dict)
            and item["data"].get("thumbnail_url")
            else None,
        ]
        if texte is not None
    )
    if isinstance(liens, list)
    else None
)